# Pipeline Component Testing
Tests each stage of the filing pipeline independently:  
**Loader → Chunker → (Sentiment/Risk → Summarizer in later phases)**

Run cells top-to-bottom. Each section is self-contained.

## 0 · Setup

In [ ]:
import sys, os, pprint

# Make app/ importable from the notebooks/ directory
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('Python   :', sys.executable)

---
## 1 · Loader — Markdown (Synthetic Corpus)
Tests `MarkdownFilingLoader` on the existing synthetic `.md` filings.

In [ ]:
from app.loader import MarkdownFilingLoader

MARKDOWN_CORPUS_DIR = os.path.join(REPO_ROOT, 'corpus', 'filings')
md_loader = MarkdownFilingLoader(corpus_dir=MARKDOWN_CORPUS_DIR)

# ── Change this to any filing in corpus/filings/ ──
FILING_ID = 'HLSR-2024'

md_docs = md_loader.load(FILING_ID)

print(f'Filing  : {FILING_ID}')
print(f'Sections loaded: {len(md_docs)}')
print()
for i, doc in enumerate(md_docs):
    print(f"  [{i}] section='{doc['metadata']['section']}'  chars={len(doc['page_content'])}")

In [ ]:
# Inspect a specific section's content
INSPECT_INDEX = 0   # change to any index from the list above

doc = md_docs[INSPECT_INDEX]
print(f"Section : {doc['metadata']['section']}")
print(f"Source  : {doc['metadata']['source']}")
print('-' * 60)
print(doc['page_content'][:800], '...' if len(doc['page_content']) > 800 else '')

---
## 2 · Loader — PDF (Real 10-K / 10-Q)

`PDFFilingLoader` uses **pdfplumber** to extract prose + tables.  
Tables are converted to Markdown pipe format and appended under a `[TABLES]` marker.

> **Set `PDF_PATH` below** to the absolute path of any 10-K or 10-Q PDF.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE: set the path to your PDF and a filing ID    ║
# ╚══════════════════════════════════════════════════════════╝

PDF_PATH    = r'C:\path\to\your\10K.pdf'   # <── change this
PDF_FILING_ID = 'MY-COMPANY-2024'          # <── change this (used as title in metadata)

# Derive PDF_DIR and filename stem for the loader
PDF_DIR  = os.path.dirname(PDF_PATH)
PDF_STEM = os.path.splitext(os.path.basename(PDF_PATH))[0]  # filename without .pdf

print(f'PDF dir  : {PDF_DIR}')
print(f'Stem     : {PDF_STEM}')
print(f'Exists   : {os.path.exists(PDF_PATH)}')

In [ ]:
from app.loader import PDFFilingLoader

pdf_loader = PDFFilingLoader(pdf_dir=PDF_DIR)

# Load using the stem as filing_id (loader appends .pdf automatically)
pdf_docs = pdf_loader.load(PDF_STEM)

print(f'Pages extracted : {len(pdf_docs)}')
pages_with_tables = [d for d in pdf_docs if d['metadata'].get('has_tables')]
print(f'Pages with tables: {len(pages_with_tables)}')
print()
for i, doc in enumerate(pdf_docs[:10]):   # show first 10 pages
    m = doc['metadata']
    print(f"  [page {m['page']:>3}] section='{m['section']}'  has_tables={m['has_tables']}  chars={len(doc['page_content'])}")

In [ ]:
# Inspect a page — view prose + extracted table side by side
INSPECT_PAGE = 1   # 1-indexed page number

target = next((d for d in pdf_docs if d['metadata']['page'] == INSPECT_PAGE), None)
if target is None:
    print(f'Page {INSPECT_PAGE} not found in extracted docs.')
else:
    m = target['metadata']
    print(f"Page     : {m['page']}")
    print(f"Section  : {m['section']}")
    print(f"Has tables: {m['has_tables']}")
    print('=' * 60)
    content = target['page_content']
    # Split prose from tables for readability
    if '[TABLES]' in content:
        prose_part, table_part = content.split('[TABLES]', 1)
        print('--- PROSE ---')
        print(prose_part[:600])
        print('\n--- TABLES (Markdown) ---')
        print(table_part[:800])
    else:
        print(content[:1000])

In [ ]:
# Section coverage summary across entire PDF
from collections import Counter

section_counts = Counter(d['metadata']['section'] for d in pdf_docs)
print('Section distribution (pages per section):')
for section, count in section_counts.most_common():
    print(f'  {section:<35} {count} page(s)')

---
## 3 · Chunker
Tests `chunk_documents` on the Markdown docs (works identically for PDF docs).

Uses `RecursiveCharacterTextSplitter` if `langchain-text-splitters` is installed,  
otherwise falls back to the original fixed-size splitter.

In [ ]:
from app.chunker import chunk_documents, _LANGCHAIN_AVAILABLE

print(f'LangChain splitter available: {_LANGCHAIN_AVAILABLE}')

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150

md_chunks = chunk_documents(md_docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f'\nMarkdown — sections in: {len(md_docs)}  →  chunks out: {len(md_chunks)}')

text_chunks  = [c for c in md_chunks if c['metadata'].get('chunk_type') != 'table']
table_chunks = [c for c in md_chunks if c['metadata'].get('chunk_type') == 'table']
print(f'  text chunks : {len(text_chunks)}')
print(f'  table chunks: {len(table_chunks)}')

sizes = [len(c['page_content']) for c in md_chunks]
print(f'\nChunk sizes — min: {min(sizes)}  max: {max(sizes)}  avg: {sum(sizes)//len(sizes)}')

In [ ]:
# Inspect individual chunks
INSPECT_CHUNK = 0   # change to any chunk index

ch = md_chunks[INSPECT_CHUNK]
print(f"Section    : {ch['metadata'].get('section')}")
print(f"Chunk index: {ch['metadata'].get('chunk_index')}")
print(f"Chunk type : {ch['metadata'].get('chunk_type', 'text')}")
print(f"Chars      : {len(ch['page_content'])}")
print('-' * 60)
print(ch['page_content'])

In [ ]:
# Overlap sanity check — confirm adjacent text chunks share content
text_only = [c for c in md_chunks if c['metadata'].get('chunk_type', 'text') == 'text']

if len(text_only) >= 2 and _LANGCHAIN_AVAILABLE:
    a = text_only[0]['page_content']
    b = text_only[1]['page_content']
    # Count shared tokens
    tokens_a = set(a.lower().split())
    tokens_b = set(b.lower().split())
    overlap_tokens = tokens_a & tokens_b
    print(f'Shared tokens between chunk 0 and chunk 1: {len(overlap_tokens)}')
    print(f'Sample overlap tokens: {list(overlap_tokens)[:10]}')
else:
    print('Not enough text chunks or LangChain unavailable — overlap check skipped.')

---
## 4 · Chunker on PDF Docs
Same chunker, now applied to PDF-extracted documents.  
Table chunks should appear as intact Markdown tables (not split mid-row).

In [ ]:
# Skip this cell if PDF was not loaded above
if 'pdf_docs' not in dir() or not pdf_docs:
    print('No PDF docs loaded — run Section 2 first.')
else:
    pdf_chunks = chunk_documents(pdf_docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    pdf_text_chunks  = [c for c in pdf_chunks if c['metadata'].get('chunk_type') != 'table']
    pdf_table_chunks = [c for c in pdf_chunks if c['metadata'].get('chunk_type') == 'table']

    print(f'PDF — pages in: {len(pdf_docs)}  →  chunks out: {len(pdf_chunks)}')
    print(f'  text chunks : {len(pdf_text_chunks)}')
    print(f'  table chunks: {len(pdf_table_chunks)}')

    if pdf_table_chunks:
        print('\n--- First table chunk (Markdown pipe format) ---')
        print(pdf_table_chunks[0]['page_content'][:600])

---
## 5 · LoaderFactory — Dynamic Source Switching
Tests the factory toggle between `markdown` and `pdf` sources.

In [ ]:
from app.loader import LoaderFactory

# Test markdown path via factory
os.environ['FILING_SOURCE'] = 'markdown'
loader = LoaderFactory.get()
print(f'Factory returned: {type(loader).__name__}')

factory_docs = loader.load('HLSR-2024')
print(f'Sections loaded : {len(factory_docs)}')
assert all('section' in d['metadata'] for d in factory_docs), 'metadata.section missing!'
assert all('source'  in d['metadata'] for d in factory_docs), 'metadata.source missing!'
print('✓ All documents have required metadata keys (section, source)')

In [ ]:
# Test pdf path via factory (expects PDF_DIR env var + PDF file present)
if 'pdf_docs' in dir() and pdf_docs:
    os.environ['FILING_SOURCE'] = 'pdf'
    os.environ['PDF_DIR'] = PDF_DIR
    pdf_factory_loader = LoaderFactory.get()
    print(f'Factory returned: {type(pdf_factory_loader).__name__}')
    pdf_factory_docs = pdf_factory_loader.load(PDF_STEM)
    print(f'Pages loaded via factory: {len(pdf_factory_docs)}')
    print('✓ PDFFilingLoader works through factory')
else:
    print('PDF not loaded — skipping factory PDF test.')

# Reset to default
os.environ['FILING_SOURCE'] = 'markdown'

---
## 6 · Individual Tool Testing
Tests every `@tool` function from `app/tools.py` in isolation — no LLM required.
All tools use rule-based logic or call `app/loader.py` / `app/sentiment_risk.py` directly.

In [ ]:
from app.tools import (
    load_filing_tool,
    extract_sections_tool,
    score_sentiment_tool,
    extract_risk_factors_tool,
    extract_financial_tables_tool,
    validate_output_tool,
    groundedness_check_tool,
    ALL_TOOLS,
)

print(f'Tools registered: {len(ALL_TOOLS)}')
for t in ALL_TOOLS:
    print(f'  {t.name:<32}  {t.description[:60]}')

In [ ]:
# ── Tool 1: load_filing_tool ──────────────────────────────────────────────────
result = load_filing_tool.invoke({"filing_id": "HLSR-2024", "source": "markdown"})
print(f'section_count : {result["section_count"]}')
print(f'first section : {result["documents"][0]["metadata"]["section"]}')
print(f'first 200 chars:\n{result["documents"][0]["page_content"][:200]}')

In [ ]:
# ── Tool 2: extract_sections_tool ────────────────────────────────────────────
docs = result["documents"]
sections_result = extract_sections_tool.invoke({
    "documents":     docs,
    "section_names": ["Business", "Risk Factors", "Results"],
})
print(f'Total docs      : {len(docs)}')
print(f'Filtered count  : {sections_result["count"]}')
print(f'Sections found  : {sections_result["sections_found"]}')

In [ ]:
# ── Tool 3: score_sentiment_tool ─────────────────────────────────────────────
sample_text = " ".join(d["page_content"] for d in docs[:3])
sentiment = score_sentiment_tool.invoke({"text": sample_text})
print(f'Positive tokens : {sentiment["positive"]}')
print(f'Negative tokens : {sentiment["negative"]}')
print(f'Uncertain tokens: {sentiment["uncertain"]}')
print(f'Tone            : {sentiment["tone"]}')

# ── Tool 4: extract_risk_factors_tool ─────────────────────────────────────────
risk_docs = sections_result["documents"]
risks = extract_risk_factors_tool.invoke({"documents": risk_docs, "top_k": 3})
print(f'\nRisk factors extracted: {len(risks)}')
for r in risks:
    print(f'  [{r["severity"].upper():>6}] {r["snippet"][:80]}  ({r["section"]})')

In [ ]:
# ── Tool 5: extract_financial_tables_tool ────────────────────────────────────
# Markdown corpus has no pipe tables, so raw_tables will be empty — expected.
financials = extract_financial_tables_tool.invoke({"documents": docs})
print('Financial extraction (Markdown corpus — no tables expected):')
for k, v in financials.items():
    if k != 'raw_tables':
        print(f'  {k:<15}: {v}')
print(f'  table_chunks    : {len(financials["raw_tables"])}')

# ── Tool 6: validate_output_tool ─────────────────────────────────────────────
mock_summary = {
    "highlights": ["Revenue grew 12% YoY. (Results)", "New product launched. (Business)"],
    "risks":      risks[:3],
    "tone":       sentiment["tone"],
    "financials": financials,
}
validation = validate_output_tool.invoke({"filing_summary": mock_summary})
print(f'\nValidation: valid={validation["valid"]}  errors={validation["errors"]}')

# ── Tool 7: groundedness_check_tool ──────────────────────────────────────────
source_text  = " ".join(d["page_content"] for d in docs)
summary_text = " ".join(mock_summary["highlights"] + [r["snippet"] for r in mock_summary["risks"]])
ground = groundedness_check_tool.invoke({"summary_text": summary_text, "source_text": source_text})
print(f'\nGroundedness: score={ground["score"]}  grounded={ground["grounded"]}  '
      f'({ground["hit_tokens"]}/{ground["total_tokens"]} tokens found in source)')

---
## 7 · Agent Node Testing
Tests each agent node function individually with a manually constructed `FilingState`.
No LLM is required — `MOCK_LLM=true` forces each agent to use its rule-based fallback.

Each node takes a partial `FilingState` and returns a partial state update (dict).

In [ ]:
import os
os.environ['MOCK_LLM'] = 'true'   # Force rule-based fallback in all agents

from app.agents import (
    orchestrator_node,
    sentiment_node,
    risk_node,
    financial_node,
    summarizer_node,
    evaluator_node,
)

# Build a minimal FilingState to pass into each node
TEST_FILING_ID = 'HLSR-2024'

# ── Step 1: Run orchestrator to get properly populated state ──────────────────
initial_state = {
    "filing_id":    TEST_FILING_ID,
    "source_docs":  [],
    "sections":     {},
    "risks":        [],
    "sources":      [],
    "errors":       [],
    "retry_count":  0,
}
orch_update = orchestrator_node(initial_state)
state = {**initial_state, **orch_update}
print(f'Orchestrator output keys: {list(orch_update.keys())}')
print(f'Sections loaded: {list(state["sections"].keys())}')

In [ ]:
# ── Sentiment Agent ───────────────────────────────────────────────────────────
sent_update = sentiment_node(state)
state = {**state, **sent_update}
print(f'Sentiment Agent output:')
print(f'  tone           : {state["tone"]}')
print(f'  tone_reasoning : {state["tone_reasoning"]}')
print()

# ── Risk Agent ────────────────────────────────────────────────────────────────
risk_update = risk_node(state)
state = {**state, "risks": state.get("risks", []) + risk_update.get("risks", [])}
print(f'Risk Agent output: {len(state["risks"])} risk(s)')
for r in state["risks"]:
    print(f'  [{r.get("severity","?").upper():>6}] {r.get("snippet","")[:80]}')

In [ ]:
# ── Financial Agent ───────────────────────────────────────────────────────────
fin_update = financial_node(state)
state = {**state, **fin_update}
print('Financial Agent output:')
for k, v in state["financials"].items():
    if k != 'raw_tables':
        print(f'  {k:<15}: {v}')
print()

# ── Summarizer Agent ──────────────────────────────────────────────────────────
sum_update = summarizer_node(state)
state = {**state, **sum_update}
print('Summarizer Agent output:')
for i, h in enumerate(state.get("highlights", []), 1):
    print(f'  Highlight {i}: {h}')
print()

# ── Evaluator Agent ───────────────────────────────────────────────────────────
eval_update = evaluator_node(state)
state = {**state, **eval_update}
print('Evaluator Agent output:')
er = state["eval_result"]
print(f'  valid        : {er["valid"]}')
print(f'  schema errors: {er["validation"]["errors"]}')
print(f'  groundedness : {er["groundedness"]["score"]} '
      f'(grounded={er["groundedness"]["grounded"]})')
print(f'  retry_count  : {state["retry_count"]}')

---
## 8 · Full Graph Run (End-to-End)
Calls `build_graph()` from `app/graph.py` and runs the complete pipeline with
`MOCK_LLM=true` (rule-based fallback). To use real LLM, set `OPENAI_API_KEY` and
`os.environ['MOCK_LLM'] = 'false'` before running.

Requires: `pip install langgraph langchain-core`

In [ ]:
os.environ['MOCK_LLM'] = 'true'   # set to 'false' + set OPENAI_API_KEY for real LLM

try:
    from app.graph import build_graph, run_pipeline
    print('langgraph imported OK')
except ImportError as e:
    print(f'langgraph not installed: {e}')
    print('Run: pip install langgraph langchain-core')
    raise

In [ ]:
# Run the full graph for a single filing
GRAPH_FILING_ID = 'HLSR-2024'

result = run_pipeline(GRAPH_FILING_ID)

print(f'filing_id  : {result["filing_id"]}')
print(f'tone       : {result["tone"]}')
print(f'\nHighlights ({len(result["highlights"])}/2):')
for h in result["highlights"]:
    print(f'  • {h}')

print(f'\nRisks ({len(result["risks"])}/3):')
for r in result["risks"]:
    print(f'  [{r.get("severity","?").upper():>6}] {r.get("snippet","")[:80]}')

print(f'\nFinancials:')
for k, v in result["financials"].items():
    if k != 'raw_tables':
        print(f'  {k:<15}: {v}')

print(f'\nEval result:')
print(f'  valid        : {result["eval_result"].get("valid")}')
print(f'  groundedness : {result["eval_result"].get("groundedness", {}).get("score")}')
print(f'  errors       : {result["errors"]}')

In [ ]:
# Batch run: all 15 filings, collect tone predictions + eval results
import json

ALL_FILINGS = [
    'HLSR-2022', 'HLSR-2023', 'HLSR-2024',
    'ACMR-2022', 'ACMR-2023', 'ACMR-2024',
    'ZYNT-2022', 'ZYNT-2023', 'ZYNT-2024',
    'NEOV-2022', 'NEOV-2023', 'NEOV-2024',
    'LUMO-2022', 'LUMO-2023', 'LUMO-2024',
]

gold_path = os.path.join(REPO_ROOT, 'evaluation', 'gold_labels.json')
gold      = json.load(open(gold_path))

rows = []
for fid in ALL_FILINGS:
    r     = run_pipeline(fid)
    pred  = r['tone']
    gold_t = gold.get(fid, {}).get('tone', 'N/A')
    match  = pred == gold_t
    rows.append({'filing': fid, 'pred': pred, 'gold': gold_t, 'match': match,
                 'valid': r['eval_result'].get('valid', False)})

hits = sum(r['match'] for r in rows)
print(f'Tone accuracy: {hits}/{len(rows)} = {hits/len(rows):.0%}\n')
print(f'{"Filing":<15} {"Pred":<12} {"Gold":<12} {"Match":<6} {"Valid"}')
print('-' * 55)
for r in rows:
    print(f'{r["filing"]:<15} {r["pred"]:<12} {r["gold"]:<12} {"✓" if r["match"] else "✗":<6} {"✓" if r["valid"] else "✗"}')

---
## 9 · MCP Server Testing
Tests the `filing-loader-mcp` server tool functions directly in-process
(no subprocess/transport needed for testing the logic).

In production the server runs as a stdio process and is registered in your
MCP client config (Claude Desktop, VS Code MCP extension, etc.).

In [ ]:
try:
    from mcp.server.fastmcp import FastMCP
    print('mcp package available ✓')
except ImportError:
    print('mcp not installed — run: pip install "mcp[cli]"')
    print('MCP tests below will import functions directly instead.')

# Import the tool functions directly (bypasses transport, tests logic only)
from mcp_server.filing_loader_server import load_filing, get_sections, chunk_filing

print('\nMCP server tools imported successfully')

In [ ]:
# ── MCP Tool: load_filing ─────────────────────────────────────────────────────
mcp_load = load_filing(filing_id='ACMR-2024', source='markdown')
print(f'load_filing  → section_count={mcp_load["section_count"]}')
print(f'             → sections={mcp_load["sections"]}')

# ── MCP Tool: get_sections ────────────────────────────────────────────────────
mcp_secs = get_sections(
    filing_id='ACMR-2024',
    section_names=['Business', 'Risk Factors'],
    source='markdown',
)
print(f'\nget_sections → count={mcp_secs["count"]}  sections_found={mcp_secs["sections_found"]}')

# ── MCP Tool: chunk_filing ────────────────────────────────────────────────────
mcp_chunks = chunk_filing(filing_id='ACMR-2024', source='markdown')
print(f'\nchunk_filing → total={mcp_chunks["total"]}  '
      f'text={len(mcp_chunks["text_chunks"])}  '
      f'table={len(mcp_chunks["table_chunks"])}')

# ── MCP registration command (for reference) ─────────────────────────────────
print('\n── To register as an MCP server ──────────────────────────────────────')
mcp_cmd = f'python -m mcp_server.filing_loader_server'
print(f'Command: {mcp_cmd}')
print(f'Add to MCP config JSON:')
print('''{
  "mcpServers": {
    "filing-loader": {
      "command": "python",
      "args": ["-m", "mcp_server.filing_loader_server"],
      "cwd": "''' + REPO_ROOT.replace('\\', '/') + '''"
    }
  }
}''')